# Subsample the simulation log

Select sample of  final surviving cells uniformly without replacement, then retain every ancestor back to the origin cell. Shared ancestors appear once, and all original columns are preserved.

Read `data/log_eden.tsv` and save the result to `data/log_eden_subsampled.tsv`. The seed makes the selection reproducible. This assumes a completed simulation with survivors, identified by the maximum `death_date`.

In [28]:
from pathlib import Path

# Support running from either eden_sim/ or the project directory.
if Path("eden_sim").is_dir():
    from eden_sim.subsample import subsample_log
    from eden_sim.log_to_fasta import convert_fasta
else:
    from subsample import subsample_log
    from log_to_fasta import convert_fasta

SIM_DIR = Path.cwd() / "eden_sim" if Path("eden_sim").is_dir() else Path.cwd()
INPUT_FILE = SIM_DIR / "data" / "log_eden.tsv"
OUTPUT_FILE = INPUT_FILE.with_name("log_eden_subsampled.tsv")
FASTA_FILE = INPUT_FILE.with_name("log_eden_subsampled.fasta")
# PROPORTION = 0.10
N = 30
SEED = 42

In [29]:
subsampled = subsample_log(INPUT_FILE, n = N, seed=SEED)
subsampled_survivors = subsampled[subsampled["death_date"].eq(subsampled["death_date"].max())]

subsampled_survivors.to_csv("temp", sep="\t", index=False)

subsampled.to_csv(OUTPUT_FILE, sep="\t", index=False)
convert_fasta("temp", FASTA_FILE)


In [30]:
sampled_survivors = int(subsampled["death_date"].eq(subsampled["death_date"].max()).sum())
ancestor_count = len(subsampled) - sampled_survivors

print(f"Sampled {sampled_survivors} final survivors ({PROPORTION:.0%}, rounded up).")
print(f"Retained {ancestor_count} ancestors; {len(subsampled)} total rows.")
print(f"Saved to {OUTPUT_FILE}")

Sampled 30 final survivors (10%, rounded up).
Retained 362 ancestors; 392 total rows.
Saved to /Users/hwijungkim/Code/bioinfo/CS189/project/eden_sim/data/log_eden_subsampled.tsv
